# AutoShop Multi-Agent Graph mit OpenAI Agents SDK

Dieses Notebook baut die bisherige Agents-SDK-Version zu einer **Multi-Agent-Graph-Architektur** um.

Die fachlichen Komponenten bleiben erhalten:

- AutoShop MCP-Server für interne Daten
- interner AutoShop-Agent mit MCP-Zugriff
- externer Enrichment-Agent ohne MCP-Zugriff
- Offert-Agent ohne MCP-Zugriff

Neu hinzu kommen:

1. ein **Request-Analysis-Agent** für Intent und Routing,
2. ein expliziter **typed Workflow State**,
3. klar definierte **Graph Nodes**,
4. **Conditional Edges** zwischen den Nodes,
5. eine deterministische Python-Orchestrierung statt einer fest verdrahteten Agentenkette.

Der Graph wird bewusst ohne LangGraph implementiert. Die Agenten stammen aus dem OpenAI Agents SDK; die Workflow-Orchestrierung bleibt transparentes Python.


## Architektur

```text
START
  |
  v
analyze_request
  |
  +---------------------------+
  | internal_search_required? |
  +-------------+-------------+
                |
             ja | nein
                v
        internal_lookup
                |
                v
        enrichment_required?
           /          \
         ja            nein
         |               |
         v               |
      enrich             |
         |               |
         +-------+-------+
                 |
                 v
          offer_required?
            /        \
          ja          nein
          |             |
          v             |
      create_offer      |
          |             |
          +------+------+
                 |
                 v
              finalize
                 |
                 v
                END
```

Die Entscheidungen werden im `WorkflowState` gespeichert. Damit müssen die Agenten ihre Zwischenresultate nicht implizit über freie Textketten "merken".


---

## Installation

Das Notebook verwendet das OpenAI Agents SDK und den bestehenden MCP-Server.


In [ ]:
%pip install -q -U openai openai-agents "mcp>=1.19,<3" pydantic

## Umgebung und MCP-Verbindung

Die bestehende `~/data/env.py` wird weiterverwendet.

Der MCP-Endpunkt wird wie bisher über den Kubernetes-Service ermittelt.

In [ ]:
%run ~/data/env.py

import subprocess
from dataclasses import dataclass, field
from typing import Any, Awaitable, Callable

from pydantic import BaseModel, Field
from openai import AsyncOpenAI

from agents import (
    Agent,
    Runner,
    set_default_openai_client,
    set_tracing_disabled,
)
from agents.mcp import (
    MCPServerStreamableHttp,
    create_static_tool_filter,
)


def get_server_url():
    server_ip = subprocess.check_output(
        "cat ~/data/server-ip",
        shell=True,
        text=True,
    ).strip()

    node_port = subprocess.check_output(
        "kubectl get service --namespace ms-mcp autoshop-mcp-server "
        "-o=jsonpath='{ .spec.ports[0].nodePort }'",
        shell=True,
        text=True,
    ).strip()

    return f"http://{server_ip}:{node_port}/mcp"


MCP_URL = get_server_url()

openai_client = AsyncOpenAI(
    api_key=OPENAI_API_KEY,
    base_url=AI_BASE_URL,
)

set_default_openai_client(
    openai_client,
    use_for_tracing=False,
)

# Für OpenAI-kompatible Endpoints ist OpenAI-Tracing nicht zwingend verfügbar.
# Bei direkter Nutzung der OpenAI Platform kann diese Zeile entfernt werden.
set_tracing_disabled(True)

print(f"MCP URL: {MCP_URL}")
print(f"Model:   {AI_MODEL}")


---

## Read-only MCP-Toolset

Nur lesende AutoShop-Tools werden an den internen Agenten exponiert. Schreibende oder destruktive Tools wie `order_delete` bleiben ausgeschlossen.


In [ ]:
READ_ONLY_TOOLS = [
    "catalog_list_items",
    "catalog_get_item",
    "customer_list_items",
    "customer_get_item",
    "order_list_items",
    "order_get_item",
]

read_only_filter = create_static_tool_filter(
    allowed_tool_names=READ_ONLY_TOOLS,
)

autoshop_mcp = MCPServerStreamableHttp(
    name="AutoShop MCP",
    params={
        "url": MCP_URL,
        "timeout": 30,
    },
    cache_tools_list=True,
    tool_filter=read_only_filter,
    max_retry_attempts=2,
)


---

# 1. Typed Routing Result

Der Request-Analysis-Agent liefert keine freie Textantwort, sondern ein strukturiertes Routing-Objekt.

Damit wird die Entscheidung des Analyse-Agenten explizit im Graph-State abgelegt.


In [ ]:
class RequestPlan(BaseModel):
    intent: str = Field(
        description="Kurze Beschreibung der Benutzerabsicht."
    )
    internal_search_required: bool = Field(
        description="Ob interne AutoShop-Daten benötigt werden."
    )
    enrichment_required: bool = Field(
        description="Ob allgemeine externe Zusatzinformationen sinnvoll sind."
    )
    offer_required: bool = Field(
        description="Ob eine Offerte erstellt werden soll."
    )
    reasoning_summary: str = Field(
        description="Kurze Begründung für das Routing."
    )


# 2. Workflow State

Alle Nodes lesen und verändern denselben `AutoShopState`.

Das ist der zentrale Unterschied zur linearen Variante: Zwischenresultate und Routing-Entscheide liegen explizit im State.


In [ ]:
@dataclass
class AutoShopState:
    customer_name: str
    user_request: str

    plan: RequestPlan | None = None

    internal_result: str | None = None
    enrichment_result: str | None = None
    offer: str | None = None

    final_output: str | None = None

    visited_nodes: list[str] = field(default_factory=list)
    errors: list[str] = field(default_factory=list)

    def mark(self, node_name: str):
        self.visited_nodes.append(node_name)


---

# 3. Spezialisten-Agenten

Der Workflow verwendet vier getrennte Agenten:

1. `request_analysis_agent` – klassifiziert den Request und bestimmt das Routing.
2. `internal_car_agent` – darf als einziger den AutoShop-MCP-Server verwenden.
3. `external_enrichment_agent` – ergänzt nur allgemeine Informationen.
4. `offer_agent` – erstellt die finale Demo-Offerte.


In [ ]:
request_analysis_agent = Agent(
    name="AutoShop Request Analysis Agent",
    model=AI_MODEL,
    output_type=RequestPlan,
    instructions="""
Du analysierst Kundenanfragen für einen AutoShop-Workflow.

Bestimme:
- intent
- internal_search_required
- enrichment_required
- offer_required
- reasoning_summary

Regeln:
- Für konkrete Fahrzeugempfehlungen, Catalog-Daten, Preise, Kunden oder Aufträge
  ist internal_search_required=true.
- enrichment_required=true, wenn allgemeine Zusatzinformationen, typische
  Vorteile, Extras oder Verkaufsargumente sinnvoll sind.
- offer_required=true, wenn der Benutzer eine Empfehlung, ein Angebot oder
  eine verkaufsorientierte Fahrzeugauswahl erwartet.
- Erfinde keine AutoShop-Daten.
- Antworte ausschliesslich über das strukturierte Output-Schema.
""",
)


internal_car_agent = Agent(
    name="AutoShop Internal Car Agent",
    model=AI_MODEL,
    instructions="""
Du bist der interne Auto-Verkaufsagent für den AutoShop.

Aufgabe:
1. Analysiere den Kundenwunsch.
2. Verwende die verfügbaren AutoShop-MCP-Tools.
3. Liste passende Fahrzeuge aus dem Catalog auf.
4. Verwende ausschliesslich Preise aus dem Catalog.
5. Wähle das am besten passende Fahrzeug aus.
6. Begründe die Auswahl kurz.
7. Gib die Antwort strukturiert als gut lesbaren Text aus.

Regeln:
- Erfinde keine Fahrzeuge, IDs, Preise oder Verfügbarkeiten.
- Shop-Daten müssen über MCP beschafft werden.
- Wenn Eigenschaften fehlen, verwende nur tatsächlich gelieferte Catalog-Daten.
- Antworte auf Deutsch.
""",
    mcp_servers=[autoshop_mcp],
)


external_enrichment_agent = Agent(
    name="External Car Enrichment Agent",
    model=AI_MODEL,
    instructions="""
Du bist eine externe KI ohne Zugriff auf den AutoShop-Catalog und ohne MCP-Zugriff.

Du darfst:
- allgemeine Zusatzinformationen liefern,
- sinnvolle Extras vorschlagen,
- typische Vorteile eines Fahrzeugtyps erklären,
- allgemeine Verkaufsargumente formulieren.

Du darfst nicht:
- Preise erfinden,
- Shop-Verfügbarkeit erfinden,
- interne IDs erfinden,
- konkrete neue Shop-Fahrzeuge erfinden.

Kennzeichne Extras als unverbindliche Vorschläge.
Antworte auf Deutsch.
""",
)


offer_agent = Agent(
    name="AutoShop Offer Agent",
    model=AI_MODEL,
    instructions="""
Du erstellst eine Demo-Offerte als Markdown.

Regeln:
- Fahrzeuge, IDs und Preise dürfen ausschliesslich aus den übergebenen
  internen AutoShop-Daten stammen.
- Externe Informationen dürfen nur als unverbindliche Zusatzinformationen
  verwendet werden.
- Erfinde keine Preise, IDs, Verfügbarkeiten oder Fahrzeuge.
- Die Offerte ist ein Education-Beispiel und nicht rechtsverbindlich.
- Antworte auf Deutsch.
""",
)


---

# 4. Graph Nodes

Jeder Node hat dieselbe Grundsignatur:

```python
async def node(state: AutoShopState) -> AutoShopState:
    ...
    return state
```

Dadurch können Nodes unabhängig getestet und später leicht ersetzt oder erweitert werden.


In [ ]:
async def analyze_request_node(state: AutoShopState) -> AutoShopState:
    state.mark("analyze_request")

    result = await Runner.run(
        request_analysis_agent,
        f"""
Kunde:
{state.customer_name}

Kundenanfrage:
{state.user_request}

Analysiere die Anfrage und bestimme das Routing für den AutoShop-Workflow.
""",
    )

    state.plan = result.final_output
    return state


async def internal_lookup_node(state: AutoShopState) -> AutoShopState:
    state.mark("internal_lookup")

    result = await Runner.run(
        internal_car_agent,
        f"""
Kundenwunsch:
{state.user_request}

Suche passende Fahrzeuge im AutoShop und gib eine begründete Empfehlung ab.
Verwende ausschliesslich Daten aus dem AutoShop-MCP-Server.
""",
    )

    state.internal_result = result.final_output
    return state


async def enrichment_node(state: AutoShopState) -> AutoShopState:
    state.mark("enrichment")

    internal_context = state.internal_result or "Keine internen Fahrzeugdaten vorhanden."

    result = await Runner.run(
        external_enrichment_agent,
        f"""
Kundenwunsch:
{state.user_request}

Interne AutoShop-Daten:
{internal_context}

Ergänze:
- mögliche sinnvolle Extras,
- typische Vorteile des empfohlenen Fahrzeugtyps,
- Punkte, auf die der Kunde achten sollte,
- kurze Verkaufsargumente.

Erfinde keine Shop-Daten und keine Preise.
""",
    )

    state.enrichment_result = result.final_output
    return state


async def offer_node(state: AutoShopState) -> AutoShopState:
    state.mark("create_offer")

    internal_context = state.internal_result or "Keine internen AutoShop-Daten vorhanden."
    enrichment_context = state.enrichment_result or "Keine externen Zusatzinformationen."

    result = await Runner.run(
        offer_agent,
        f"""
Erstelle eine Markdown-Offerte.

Kunde:
{state.customer_name}

Kundenwunsch:
{state.user_request}

Interne AutoShop-Daten:
{internal_context}

Externe Zusatzinformationen:
{enrichment_context}

Verwende exakt diese Struktur:

# Offerte

## Kunde

## Kundenwunsch

## Empfohlenes Fahrzeug

## Preis gemäss AutoShop-Catalog

## Empfohlene Extras

## Begründung

## Hinweis

Der Hinweis muss enthalten:
Diese Offerte ist ein Education-Beispiel und nicht rechtsverbindlich.
""",
    )

    state.offer = result.final_output
    return state


async def finalize_node(state: AutoShopState) -> AutoShopState:
    state.mark("finalize")

    if state.offer:
        state.final_output = state.offer
    elif state.enrichment_result:
        state.final_output = state.enrichment_result
    elif state.internal_result:
        state.final_output = state.internal_result
    elif state.plan:
        state.final_output = (
            f"Intent: {state.plan.intent}\n\n"
            f"{state.plan.reasoning_summary}"
        )
    else:
        state.final_output = "Kein Resultat verfügbar."

    return state


---

# 5. Conditional Edges

Die Routing-Funktionen enthalten keine LLM-Aufrufe. Sie werten nur den bereits vorhandenen State aus.

Damit ist der Kontrollfluss deterministisch und testbar.


In [ ]:
def route_after_analysis(state: AutoShopState) -> str:
    if state.plan is None:
        return "finalize"

    if state.plan.internal_search_required:
        return "internal_lookup"

    if state.plan.enrichment_required:
        return "enrichment"

    if state.plan.offer_required:
        return "create_offer"

    return "finalize"


def route_after_internal_lookup(state: AutoShopState) -> str:
    if state.plan and state.plan.enrichment_required:
        return "enrichment"

    if state.plan and state.plan.offer_required:
        return "create_offer"

    return "finalize"


def route_after_enrichment(state: AutoShopState) -> str:
    if state.plan and state.plan.offer_required:
        return "create_offer"

    return "finalize"


---

# 6. Minimaler Async Graph Runner

Der Graph-Runner kennt:

- Nodes
- statische Edges
- Conditional Edges
- Start-Node
- End-Node

Damit bleibt die Workflow-Engine klein genug, um die Mechanik im Notebook vollständig nachvollziehen zu können.


In [ ]:
NodeFunction = Callable[[AutoShopState], Awaitable[AutoShopState]]
RouterFunction = Callable[[AutoShopState], str]


class AsyncStateGraph:
    END = "__end__"

    def __init__(self):
        self.nodes: dict[str, NodeFunction] = {}
        self.edges: dict[str, str] = {}
        self.conditional_edges: dict[str, RouterFunction] = {}
        self.entry_point: str | None = None

    def add_node(self, name: str, func: NodeFunction):
        self.nodes[name] = func

    def set_entry_point(self, name: str):
        self.entry_point = name

    def add_edge(self, source: str, target: str):
        self.edges[source] = target

    def add_conditional_edges(self, source: str, router: RouterFunction):
        self.conditional_edges[source] = router

    async def run(self, state: AutoShopState) -> AutoShopState:
        if self.entry_point is None:
            raise RuntimeError("Kein Entry Point definiert.")

        current = self.entry_point

        while current != self.END:
            if current not in self.nodes:
                raise RuntimeError(f"Unbekannter Graph-Node: {current}")

            node = self.nodes[current]

            try:
                state = await node(state)
            except Exception as exc:
                state.errors.append(f"{current}: {type(exc).__name__}: {exc}")
                raise

            if current in self.conditional_edges:
                current = self.conditional_edges[current](state)
            elif current in self.edges:
                current = self.edges[current]
            else:
                current = self.END

        return state


---

# 7. Graph zusammensetzen

Hier wird die Topologie des Workflows explizit definiert.


In [ ]:
graph = AsyncStateGraph()

graph.add_node("analyze_request", analyze_request_node)
graph.add_node("internal_lookup", internal_lookup_node)
graph.add_node("enrichment", enrichment_node)
graph.add_node("create_offer", offer_node)
graph.add_node("finalize", finalize_node)

graph.set_entry_point("analyze_request")

graph.add_conditional_edges(
    "analyze_request",
    route_after_analysis,
)

graph.add_conditional_edges(
    "internal_lookup",
    route_after_internal_lookup,
)

graph.add_conditional_edges(
    "enrichment",
    route_after_enrichment,
)

graph.add_edge(
    "create_offer",
    "finalize",
)

graph.add_edge(
    "finalize",
    AsyncStateGraph.END,
)


---

# 8. Multi-Agent-Graph ausführen

Die MCP-Verbindung wird einmal für den gesamten Workflow geöffnet.

Der Graph entscheidet danach anhand des `RequestPlan`, welche Nodes tatsächlich ausgeführt werden.


In [ ]:
async def car_sales_graph(
    customer_name: str,
    user_request: str,
) -> AutoShopState:

    initial_state = AutoShopState(
        customer_name=customer_name,
        user_request=user_request,
    )

    async with autoshop_mcp:
        final_state = await graph.run(initial_state)

    return final_state


---

# 9. Beispiel

Bei dieser Anfrage sollte der Graph typischerweise folgende Route wählen:

```text
analyze_request
→ internal_lookup
→ enrichment
→ create_offer
→ finalize
```


In [ ]:
from IPython.display import Markdown, display

state = await car_sales_graph(
    customer_name="Max Muster",
    user_request="Ich suche ein günstiges Auto mit viel Platz für die Familie.",
)

display(Markdown(state.final_output))


## Graph-State und ausgeführte Route anzeigen

Damit ist sichtbar, welche Nodes tatsächlich ausgeführt wurden und welche Routing-Entscheidung der Analyse-Agent getroffen hat.


In [ ]:
print("Besuchte Nodes:")
print(" -> ".join(state.visited_nodes))

print("\nRouting:")
print(state.plan.model_dump() if state.plan else None)

print("\nFehler:")
print(state.errors)


## Zwischenresultate anzeigen


In [ ]:
if state.internal_result:
    display(Markdown("## Internes AutoShop-Resultat"))
    display(Markdown(state.internal_result))

if state.enrichment_result:
    display(Markdown("## Externes Enrichment"))
    display(Markdown(state.enrichment_result))

if state.offer:
    display(Markdown("## Generierte Offerte"))
    display(Markdown(state.offer))


---

# 10. Beispiel für einen anderen Graph-Pfad

Mit einer Anfrage, die keine Offerte benötigt, kann der Analyse-Agent einen kürzeren Pfad wählen.

Beispiel:

```python
info_state = await car_sales_graph(
    customer_name="Max Muster",
    user_request="Welche allgemeinen Vorteile hat ein Familien-SUV?",
)
```

Abhängig vom Routing kann daraus beispielsweise werden:

```text
analyze_request
→ enrichment
→ finalize
```

Damit werden interne MCP-Abfragen vermieden, wenn sie fachlich nicht nötig sind.


---

# 11. Unterschiede zur linearen Agents-SDK-Version

| Bereich | Lineare Version | Multi-Agent Graph |
|---|---|---|
| Ablauf | feste Python-Sequenz | explizite Nodes und Edges |
| Routing | immer gleiche Reihenfolge | Conditional Edges |
| Zustand | lokale Variablen | zentraler `AutoShopState` |
| Analyse | implizit in Agenten | eigener Request-Analysis-Agent |
| MCP | interner Agent | unverändert interner Agent |
| Enrichment | immer ausgeführt | nur wenn erforderlich |
| Offerte | immer ausgeführt | nur wenn erforderlich |
| Debugging | Agentenresultate | State + `visited_nodes` |
| Erweiterbarkeit | lineare Anpassungen | neue Nodes/Edges |
| Graph Library | keine | keine; eigener kleiner Runner |

Die Agenten bleiben für probabilistische Aufgaben zuständig.  
Der Graph kontrolliert den deterministischen Workflow.


---

# 12. Sinnvolle nächste Erweiterungen

Die Architektur lässt sich ohne grundlegenden Umbau erweitern:

```text
analyze_request
      |
      v
internal_lookup
      |
      v
validate_internal_data
      |
   +--+------------------+
   |                     |
complete             incomplete
   |                     |
   |                     v
   |                enrichment
   |                     |
   +----------+----------+
              |
              v
         create_offer
              |
              v
        validate_offer
          /       \
        OK        Retry
        |           |
        v           |
     finalize <-----+
```

Mögliche zusätzliche Nodes:

- `validate_internal_data`
- `customer_lookup`
- `order_lookup`
- `validate_offer`
- `human_approval`
- Retry-/Fallback-Pfade
- parallele interne Lookups mit `asyncio.gather`
- persistenter Workflow-State
- Agent-/Tool-Tracing

Für das Trainingsbeispiel bleibt die vorliegende Version bewusst kompakt und zeigt die Kernprinzipien einer Multi-Agent-Graph-Architektur ohne zusätzliches Graph-Framework.
